In [1]:
##%% ============================================================
# Cell 1 - Kaggle Environment Setup
# ============================================================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

import sys
import gc
import json
import math
import random
import shutil
import subprocess
import importlib.util
from pathlib import Path
from glob import glob
from dataclasses import dataclass, asdict
from collections import Counter
from typing import Optional

def install_if_missing(package_name, import_name=None):
    if import_name is None:
        import_name = package_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name, "--quiet"])

install_if_missing("albumentations", "albumentations")
install_if_missing("timm", "timm")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as cp
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

try:
    import cv2
    CV2_AVAILABLE = True
except Exception:
    CV2_AVAILABLE = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 80)
print("Cell 1 - Kaggle Environment Setup")
print("=" * 80)
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")

if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    torch.backends.cudnn.benchmark = True

gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


Cell 1 - Kaggle Environment Setup
Python: 3.12.12
PyTorch: 2.10.0+cu128
Device: cuda
GPU: Tesla T4
VRAM: 15.64 GB


In [2]:
##%% ============================================================
# Cell 2 - Kaggle Configuration (CORRECTED PATHS)
# ============================================================
INPUT_DIR = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"

CHECKPOINT_DIR = "/kaggle/input/datasets/rawanwalidmostafa/last-saved-for-test-after-local-run"
DATASET_BASE = "/kaggle/input/datasets/weitianqi/plantseg"

IMAGES_DIR = os.path.join(DATASET_BASE, "images")
MASKS_DIR = os.path.join(DATASET_BASE, "masks")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("Verifying paths...")
print("=" * 80)
print(f"Checkpoint directory exists: {os.path.exists(CHECKPOINT_DIR)}")
print(f"Images directory exists: {os.path.exists(IMAGES_DIR)}")
print(f"Masks directory exists: {os.path.exists(MASKS_DIR)}")

if os.path.exists(CHECKPOINT_DIR):
    print(f"\nFiles in checkpoint directory:")
    for f in os.listdir(CHECKPOINT_DIR):
        print(f"  {f}")

if os.path.exists(IMAGES_DIR):
    for split in ["train", "val", "test"]:
        split_dir = os.path.join(IMAGES_DIR, split)
        if os.path.exists(split_dir):
            num_files = len(os.listdir(split_dir))
            print(f"\n{split} images: {num_files}")

@dataclass
class CFG:
    PROJECT_NAME: str = "Plant Diagnosis with Lesion Segmentation"
    DATASET_NAME: str = "PlantSeg"
    
    INPUT_DIR: str = INPUT_DIR
    OUTPUT_DIR: str = OUTPUT_DIR
    CHECKPOINT_DIR: str = CHECKPOINT_DIR
    
    IMGS: str = IMAGES_DIR
    MASKS: str = MASKS_DIR
    
    SPLITS: tuple = ("train", "val", "test")
    IMAGE_EXTS: tuple = (".jpg", ".jpeg", ".png")
    MASK_EXTS: tuple = (".png", ".jpg", ".jpeg")
    
    SEED: int = 42
    CUDNN_BENCHMARK: bool = True
    
    IMG_SIZE: int = 320
    NUM_CLASSES: int = 2
    BATCH_SIZE: int = 8
    NUM_WORKERS: int = 0
    PIN_MEMORY: bool = True
    
    AMP: bool = True
    
    DEFAULT_THRESHOLD: float = 0.5
    THRESHOLD_GRID_START: float = 0.10
    THRESHOLD_GRID_END: float = 0.90
    THRESHOLD_GRID_STEP: float = 0.05
    
    POST_PROCESS: bool = True
    POST_MIN_AREA: int = 25
    POST_KERNEL_SIZE: int = 3
    
    BEST_MODEL_PATH: str = str(Path(CHECKPOINT_DIR) / "best_model.pth")
    LAST_MODEL_PATH: str = str(Path(CHECKPOINT_DIR) / "last_saved_model.pth")
    MODEL_STATE_PATH: str = str(Path(CHECKPOINT_DIR) / "plant_disease_model_state_dict.pth")

cfg = CFG()

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = cfg.CUDNN_BENCHMARK

seed_everything(cfg.SEED)

THRESHOLD_GRID = [
    round(float(x), 2)
    for x in np.arange(
        cfg.THRESHOLD_GRID_START,
        cfg.THRESHOLD_GRID_END + 1e-9,
        cfg.THRESHOLD_GRID_STEP
    )
]

print("=" * 80)
print("Cell 2 - Kaggle Configuration (PATHS VERIFIED)")
print("=" * 80)
print(f"Input Directory: {INPUT_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Checkpoint Directory: {CHECKPOINT_DIR}")
print(f"Images Directory: {IMAGES_DIR}")
print(f"Masks Directory: {MASKS_DIR}")
print(f"Device: {DEVICE}")


Verifying paths...
Checkpoint directory exists: True
Images directory exists: False
Masks directory exists: False

Files in checkpoint directory:
  last_saved_model.pth
Cell 2 - Kaggle Configuration (PATHS VERIFIED)
Input Directory: /kaggle/input
Output Directory: /kaggle/working
Checkpoint Directory: /kaggle/input/datasets/rawanwalidmostafa/last-saved-for-test-after-local-run
Images Directory: /kaggle/input/datasets/weitianqi/plantseg/images
Masks Directory: /kaggle/input/datasets/weitianqi/plantseg/masks
Device: cuda


In [3]:
##%% ============================================================
# Cell 3 - Helper Functions
# ============================================================
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

def denormalize_tensor(img_t):
    mean = torch.tensor(MEAN).view(3, 1, 1).to(img_t.device)
    std = torch.tensor(STD).view(3, 1, 1).to(img_t.device)
    return (img_t * std + mean).clamp(0, 1)

def tensor_to_uint8_image(img_t):
    img = denormalize_tensor(img_t.detach().cpu())
    img = img.permute(1, 2, 0).numpy()
    return (img * 255).clip(0, 255).astype(np.uint8)

def list_files_with_exts(folder, exts):
    folder = Path(folder)
    files = []
    if not folder.exists():
        return []
    for ext in exts:
        files.extend(folder.glob(f"*{ext}"))
        files.extend(folder.glob(f"*{ext.upper()}"))
    return sorted([str(f) for f in files])

def find_mask_for_image(img_path, mask_dir, mask_exts):
    img_path = Path(img_path)
    mask_dir = Path(mask_dir)
    stem = img_path.stem
    for ext in mask_exts:
        p1 = mask_dir / f"{stem}{ext}"
        p2 = mask_dir / f"{stem}{ext.upper()}"
        if p1.exists():
            return str(p1)
        if p2.exists():
            return str(p2)
    return None

def extract_disease_name_from_filename(img_path):
    return Path(img_path).stem.rsplit("_", 1)[0].replace("_", " ")

def make_overlay(image, mask, color=(255, 0, 0), alpha=0.60):
    overlay = image.copy()
    mask = mask.astype(bool)
    color = np.array(color, dtype=np.float32)
    overlay[mask] = (overlay[mask] * (1 - alpha) + color * alpha).astype(np.uint8)
    return overlay

def autocast_context():
    return torch.amp.autocast(
        "cuda",
        enabled=(cfg.AMP and DEVICE.type == "cuda")
    )

print("=" * 80)
print("Cell 3 - Helper Functions")
print("=" * 80)
print("Helper functions loaded")


Cell 3 - Helper Functions
Helper functions loaded


In [4]:
##%% ============================================================
# Cell 4 - Dataset Structure and Manifest
# ============================================================
print("=" * 80)
print("Cell 4 - Dataset Structure Analysis")
print("=" * 80)

for split in cfg.SPLITS:
    img_dir = os.path.join(cfg.IMGS, split)
    mask_dir = os.path.join(cfg.MASKS, split)
    
    if os.path.exists(img_dir) and os.path.exists(mask_dir):
        num_imgs = len(list_files_with_exts(img_dir, cfg.IMAGE_EXTS))
        num_masks = len(list_files_with_exts(mask_dir, cfg.MASK_EXTS))
        print(f"{split}: {num_imgs} images, {num_masks} masks")
    else:
        print(f"{split}: directory not found")


Cell 4 - Dataset Structure Analysis
train: directory not found
val: directory not found
test: directory not found


In [5]:
##%% ============================================================
# Cell 5 - Metrics and Segmentation Meter
# ============================================================
class SegmentationMeter:
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.tp = 0
        self.fp = 0
        self.fn = 0
        self.tn = 0
    
    def update(self, preds, targets):
        preds = preds.long()
        targets = targets.long()
        self.tp += ((preds == 1) & (targets == 1)).sum().item()
        self.fp += ((preds == 1) & (targets == 0)).sum().item()
        self.fn += ((preds == 0) & (targets == 1)).sum().item()
        self.tn += ((preds == 0) & (targets == 0)).sum().item()
    
    def compute(self):
        eps = 1e-6
        precision = self.tp / (self.tp + self.fp + eps)
        recall = self.tp / (self.tp + self.fn + eps)
        return {
            "iou": self.tp / (self.tp + self.fp + self.fn + eps),
            "dice": (2 * self.tp) / (2 * self.tp + self.fp + self.fn + eps),
            "accuracy": (self.tp + self.tn) / (self.tp + self.tn + self.fp + self.fn + eps),
            "precision": precision,
            "recall": recall,
            "specificity": self.tn / (self.tn + self.fp + eps),
            "f2": (5 * precision * recall) / (4 * precision + recall + eps),
            "tp": self.tp,
            "fp": self.fp,
            "fn": self.fn,
            "tn": self.tn,
        }

def logits_to_probs(logits):
    return torch.softmax(logits, dim=1)[:, 1]

def probs_to_preds(probs, threshold=0.5):
    return (probs >= threshold).long()

def compute_per_sample_metrics(pred_mask, true_mask):
    pred = pred_mask.astype(np.uint8)
    true = true_mask.astype(np.uint8)
    tp = ((pred == 1) & (true == 1)).sum()
    fp = ((pred == 1) & (true == 0)).sum()
    fn = ((pred == 0) & (true == 1)).sum()
    tn = ((pred == 0) & (true == 0)).sum()
    eps = 1e-6
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    return {
        "iou": float(tp / (tp + fp + fn + eps)),
        "dice": float((2 * tp) / (2 * tp + fp + fn + eps)),
        "accuracy": float((tp + tn) / (tp + tn + fp + fn + eps)),
        "precision": float(precision),
        "recall": float(recall),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
    }

print("=" * 80)
print("Cell 5 - Metrics")
print("=" * 80)
print("Metrics ready")


Cell 5 - Metrics
Metrics ready


In [6]:
##%% ============================================================
# Cell 6 - Model Architecture
# ============================================================
import timm

class SEBlock(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        hidden = max(ch // r, 4)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(ch, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, ch, bias=False),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        b, c, _, _ = x.shape
        y = self.avg(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, 1, bias=False),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, 1, bias=False),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, 1, bias=False),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, g, x):
        psi = self.relu(self.W_g(g) + self.W_x(x))
        psi = self.psi(psi)
        return x * psi

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.att = AttentionGate(F_g=in_ch // 2, F_l=skip_ch, F_int=max(in_ch // 4, 4))
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch // 2 + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            SEBlock(out_ch)
        )
    
    def forward(self, x, skip):
        x_up = self.up(x)
        if skip.shape[2:] != x_up.shape[2:]:
            skip = F.interpolate(skip, size=x_up.shape[2:], mode="bilinear", align_corners=False)
        skip_att = self.att(g=x_up, x=skip)
        out = torch.cat([x_up, skip_att], dim=1)
        return self.conv(out)

class EfficientNetAttentionUNet(nn.Module):
    def __init__(self, backbone='efficientnet_b4', out_ch=2, pretrained=False, use_checkpoint=False):
        super().__init__()
        self.use_checkpoint = use_checkpoint
        
        self.encoder = timm.create_model(
            backbone,
            pretrained=pretrained,
            features_only=True,
            out_indices=(0, 1, 2, 3, 4)
        )
        
        with torch.no_grad():
            _probe = torch.zeros(1, 3, 64, 64)
            _feats = self.encoder(_probe)
            enc_chs = [f.shape[1] for f in _feats]
            del _probe, _feats
        
        self.aspp = nn.Sequential(
            nn.Conv2d(enc_chs[4], enc_chs[4], 3, padding=2, dilation=2, bias=False),
            nn.BatchNorm2d(enc_chs[4]),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.3)
        )
        
        dec_ch = 256
        self.d4 = DecoderBlock(in_ch=enc_chs[4], skip_ch=enc_chs[3], out_ch=dec_ch)
        self.d3 = DecoderBlock(in_ch=dec_ch, skip_ch=enc_chs[2], out_ch=128)
        self.d2 = DecoderBlock(in_ch=128, skip_ch=enc_chs[1], out_ch=64)
        self.d1 = DecoderBlock(in_ch=64, skip_ch=enc_chs[0], out_ch=32)
        
        self.out = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, out_ch, 1)
        )
        
        self.aux_d3 = nn.Conv2d(128, out_ch, 1)
        self.aux_d4 = nn.Conv2d(dec_ch, out_ch, 1)
        
        self._init_decoder_weights()
    
    def _init_decoder_weights(self):
        encoder_modules = set(self.encoder.modules())
        for m in self.modules():
            if m in encoder_modules:
                continue
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if getattr(m, "bias", None) is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def _ckpt(self, module, *args):
        if self.use_checkpoint and self.training:
            return cp.checkpoint(module, *args, use_reentrant=False)
        return module(*args)
    
    def forward(self, x):
        features = self.encoder(x)
        e1, e2, e3, e4, e5 = features
        b = self._ckpt(self.aspp, e5)
        d4 = self.d4(b, e4)
        d3 = self.d3(d4, e3)
        d2 = self.d2(d3, e2)
        d1 = self.d1(d2, e1)
        out = self.out(d1)
        out = F.interpolate(out, size=x.shape[2:], mode="bilinear", align_corners=False)
        
        if self.training:
            aux3 = F.interpolate(self.aux_d3(d3), size=x.shape[2:], mode="bilinear", align_corners=False)
            aux4 = F.interpolate(self.aux_d4(d4), size=x.shape[2:], mode="bilinear", align_corners=False)
            return out, aux3, aux4
        return out

model = EfficientNetAttentionUNet(
    backbone='efficientnet_b4',
    out_ch=cfg.NUM_CLASSES,
    pretrained=False,
    use_checkpoint=False
).to(DEVICE)

print("=" * 80)
print("Cell 6 - Model Architecture")
print("=" * 80)
print("EfficientNet-B4 Attention U-Net created")


Cell 6 - Model Architecture
EfficientNet-B4 Attention U-Net created


In [7]:
##%% ============================================================
# Cell 7 - Load Model Checkpoint (FIXED)
# ============================================================
def load_checkpoint(path, model, device=None):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    
    print(f"Loading checkpoint from: {path}")
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    
    epoch = checkpoint.get("epoch", 0)
    metrics = checkpoint.get("metrics", {})
    
    return epoch, metrics

print("=" * 80)
print("Cell 7 - Load Model Checkpoint")
print("=" * 80)

available_models = []
if os.path.exists(cfg.CHECKPOINT_DIR):
    files = os.listdir(cfg.CHECKPOINT_DIR)
    model_files = [f for f in files if f.endswith(".pth")]
    print(f"Available model files: {model_files}")
    available_models = model_files

best_model_path = None

if "best_model.pth" in available_models:
    best_model_path = cfg.BEST_MODEL_PATH
elif "last_saved_model.pth" in available_models:
    best_model_path = cfg.LAST_MODEL_PATH
elif len(available_models) > 0:
    best_model_path = os.path.join(cfg.CHECKPOINT_DIR, available_models[0])
else:
    raise FileNotFoundError(f"No model files found in {cfg.CHECKPOINT_DIR}")

print(f"Loading model: {best_model_path}")

if not os.path.exists(best_model_path):
    raise FileNotFoundError(f"Model checkpoint not found at {best_model_path}")

epoch_loaded, metrics_loaded = load_checkpoint(best_model_path, model, DEVICE)
model.eval()

print(f"Model loaded successfully (epoch {epoch_loaded})")
if "iou" in metrics_loaded:
    print(f"Validation IoU: {metrics_loaded['iou']:.4f}")
if "dice" in metrics_loaded:
    print(f"Validation Dice: {metrics_loaded['dice']:.4f}")

gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


Cell 7 - Load Model Checkpoint
Available model files: ['last_saved_model.pth']
Loading model: /kaggle/input/datasets/rawanwalidmostafa/last-saved-for-test-after-local-run/last_saved_model.pth
Loading checkpoint from: /kaggle/input/datasets/rawanwalidmostafa/last-saved-for-test-after-local-run/last_saved_model.pth
Model loaded successfully (epoch 18)
Validation IoU: 0.6598
Validation Dice: 0.7950


In [8]:
##%% ============================================================
# Cell 8 - Augmentation Transforms
# ============================================================
test_transform = A.Compose([
    A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

print("=" * 80)
print("Cell 8 - Augmentation Transforms")
print("=" * 80)
print("Test transform ready")


Cell 8 - Augmentation Transforms
Test transform ready


In [11]:
##%% ============================================================
# Cell 8.5 - Explore Dataset Structure
# ============================================================
print("=" * 80)
print("Exploring Dataset Structure")
print("=" * 80)

dataset_base = "/kaggle/input/datasets/weitianqi/plantseg"

print(f"\nDirectory tree of {dataset_base}:")
for root, dirs, files in os.walk(dataset_base):
    level = root.replace(dataset_base, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for f in files[:5]:
        print(f'{subindent}{f}')
    if len(files) > 5:
        print(f'{subindent}... and {len(files) - 5} more files')
    if level > 3:
        break

print("\n" + "=" * 80)
print("Full directory listing:")
print("=" * 80)

if os.path.exists(dataset_base):
    for item in os.listdir(dataset_base):
        full_path = os.path.join(dataset_base, item)
        if os.path.isdir(full_path):
            print(f"[DIR] {item}")
            try:
                subitems = os.listdir(full_path)
                print(f"      Contains: {subitems[:10]}")
            except Exception as e:
                print(f"      Error reading: {e}")
        else:
            print(f"[FILE] {item}")


Exploring Dataset Structure

Directory tree of /kaggle/input/datasets/weitianqi/plantseg:
plantseg/
  plantsegv2/
    Metadatav2.csv
    coco_annotations.json
    annotations/
      val/
        wheat_head_scab_Baidu_0057.png
        ginger_sheath_blight_28.png
        coffee_brown_eye_spot_Bing_0003.png
        grapevine_leafroll_disease_41.png
        coffee_leaf_rust_google_0023.png
        ... and 1242 more files
      test/
        blueberry_scorch_Bing_0221.png
        cucumber_angular_leaf_spot_243.png
        wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0202.png
        rice_sheath_blight_57.png
        cucumber_bacterial_wilt_google_0024.png
        ... and 2290 more files
      train/
        wheat_head_scab_Baidu_0344.png
        wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0002.png
        potato_late_blight_google_0049.png
        rice_sheath_blight_google_0089.png
        banana_bunchy_top_Bing_0223.png
        ... and 7911 more files
    images/
      val/
        

In [12]:
##%% ============================================================
# Cell 2 - Kaggle Configuration (CORRECT PATHS)
# ============================================================
INPUT_DIR = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"

CHECKPOINT_DIR = "/kaggle/input/datasets/rawanwalidmostafa/last-saved-for-test-after-local-run"

DATASET_BASE = "/kaggle/input/datasets/weitianqi/plantseg/plantsegv2"

IMAGES_DIR = os.path.join(DATASET_BASE, "images")
MASKS_DIR = os.path.join(DATASET_BASE, "annotations")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("Verifying paths...")
print("=" * 80)
print(f"Checkpoint directory exists: {os.path.exists(CHECKPOINT_DIR)}")
print(f"Images directory exists: {os.path.exists(IMAGES_DIR)}")
print(f"Masks directory exists: {os.path.exists(MASKS_DIR)}")

if os.path.exists(CHECKPOINT_DIR):
    print(f"\nFiles in checkpoint directory:")
    for f in os.listdir(CHECKPOINT_DIR):
        print(f"  {f}")

if os.path.exists(IMAGES_DIR):
    for split in ["train", "val", "test"]:
        split_dir = os.path.join(IMAGES_DIR, split)
        if os.path.exists(split_dir):
            num_files = len(os.listdir(split_dir))
            print(f"\n{split} images: {num_files}")

@dataclass
class CFG:
    PROJECT_NAME: str = "Plant Diagnosis with Lesion Segmentation"
    DATASET_NAME: str = "PlantSeg"
    
    INPUT_DIR: str = INPUT_DIR
    OUTPUT_DIR: str = OUTPUT_DIR
    CHECKPOINT_DIR: str = CHECKPOINT_DIR
    
    IMGS: str = IMAGES_DIR
    MASKS: str = MASKS_DIR
    
    SPLITS: tuple = ("train", "val", "test")
    IMAGE_EXTS: tuple = (".jpg", ".jpeg", ".png")
    MASK_EXTS: tuple = (".png", ".jpg", ".jpeg")
    
    SEED: int = 42
    CUDNN_BENCHMARK: bool = True
    
    IMG_SIZE: int = 320
    NUM_CLASSES: int = 2
    BATCH_SIZE: int = 8
    NUM_WORKERS: int = 0
    PIN_MEMORY: bool = True
    
    AMP: bool = True
    
    DEFAULT_THRESHOLD: float = 0.5
    THRESHOLD_GRID_START: float = 0.10
    THRESHOLD_GRID_END: float = 0.90
    THRESHOLD_GRID_STEP: float = 0.05
    
    POST_PROCESS: bool = True
    POST_MIN_AREA: int = 25
    POST_KERNEL_SIZE: int = 3
    
    BEST_MODEL_PATH: str = str(Path(CHECKPOINT_DIR) / "best_model.pth")
    LAST_MODEL_PATH: str = str(Path(CHECKPOINT_DIR) / "last_saved_model.pth")
    MODEL_STATE_PATH: str = str(Path(CHECKPOINT_DIR) / "plant_disease_model_state_dict.pth")

cfg = CFG()

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = cfg.CUDNN_BENCHMARK

seed_everything(cfg.SEED)

THRESHOLD_GRID = [
    round(float(x), 2)
    for x in np.arange(
        cfg.THRESHOLD_GRID_START,
        cfg.THRESHOLD_GRID_END + 1e-9,
        cfg.THRESHOLD_GRID_STEP
    )
]

print("=" * 80)
print("Cell 2 - Kaggle Configuration (PATHS VERIFIED)")
print("=" * 80)
print(f"Input Directory: {INPUT_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Checkpoint Directory: {CHECKPOINT_DIR}")
print(f"Images Directory: {IMAGES_DIR}")
print(f"Masks Directory: {MASKS_DIR}")
print(f"Device: {DEVICE}")


Verifying paths...
Checkpoint directory exists: True
Images directory exists: True
Masks directory exists: True

Files in checkpoint directory:
  last_saved_model.pth

train images: 7916

val images: 1247

test images: 2295
Cell 2 - Kaggle Configuration (PATHS VERIFIED)
Input Directory: /kaggle/input
Output Directory: /kaggle/working
Checkpoint Directory: /kaggle/input/datasets/rawanwalidmostafa/last-saved-for-test-after-local-run
Images Directory: /kaggle/input/datasets/weitianqi/plantseg/plantsegv2/images
Masks Directory: /kaggle/input/datasets/weitianqi/plantseg/plantsegv2/annotations
Device: cuda


In [13]:
##%% ============================================================
# Cell 9 - Dataset Class
# ============================================================
class PlantSegDataset(Dataset):
    def __init__(self, split="test", transform=None, manifest_path=None, use_clean_only=True, strict_masks=False):
        self.split = split
        self.transform = transform
        self.manifest_path = manifest_path
        self.use_clean_only = use_clean_only
        self.strict_masks = strict_masks
        
        self.img_dir = os.path.join(cfg.IMGS, split)
        self.mask_dir = os.path.join(cfg.MASKS, split)
        
        self.samples = self._load_samples()
        print(f"{split}: {len(self.samples)} samples loaded")
    
    def _load_samples(self):
        samples = []
        
        if self.manifest_path is not None and os.path.exists(self.manifest_path):
            df = pd.read_csv(self.manifest_path)
            
            if self.use_clean_only and "use_for_training" in df.columns:
                df = df[df["use_for_training"] == True].copy()
            
            for _, row in df.iterrows():
                mask_path = row.get("mask_path", None)
                
                if isinstance(mask_path, float) and np.isnan(mask_path):
                    mask_path = None
                
                samples.append({
                    "image_path": row["image_path"],
                    "mask_path": mask_path,
                    "image_name": row.get("image_name", Path(row["image_path"]).name),
                    "disease_name": row.get("disease_name", extract_disease_name_from_filename(row["image_path"]))
                })
        else:
            images = list_files_with_exts(self.img_dir, cfg.IMAGE_EXTS)
            
            for img_path in images:
                samples.append({
                    "image_path": img_path,
                    "mask_path": find_mask_for_image(img_path, self.mask_dir, cfg.MASK_EXTS),
                    "image_name": Path(img_path).name,
                    "disease_name": extract_disease_name_from_filename(img_path)
                })
        
        return samples
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        image = np.array(Image.open(sample["image_path"]).convert("RGB"))
        h, w = image.shape[:2]
        
        mask_path = sample["mask_path"]
        
        if mask_path and isinstance(mask_path, str) and os.path.exists(mask_path):
            mask = np.array(Image.open(mask_path).convert("L"))
            
            if mask.shape[:2] != (h, w):
                mask = np.array(Image.fromarray(mask).resize((w, h), Image.NEAREST))
            
            mask = (mask > 0).astype(np.uint8)
        else:
            if self.strict_masks:
                raise FileNotFoundError(f"Mask not found for {sample['image_path']}")
            mask = np.zeros((h, w), dtype=np.uint8)
        
        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image = aug["image"]
            mask = aug["mask"]
        
        if not torch.is_tensor(image):
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
        
        if not torch.is_tensor(mask):
            mask = torch.from_numpy(mask)
        
        return image, mask.long()

test_ds = PlantSegDataset(
    split="test",
    transform=test_transform,
    manifest_path=None,
    use_clean_only=False,
    strict_masks=False
)

test_loader = DataLoader(
    test_ds,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=cfg.PIN_MEMORY,
    drop_last=False
)

print("=" * 80)
print("Cell 9 - Dataset and DataLoader")
print("=" * 80)
print(f"Test batches: {len(test_loader)}")


test: 2295 samples loaded
Cell 9 - Dataset and DataLoader
Test batches: 287


In [16]:
##%% ============================================================
# Cell 10 - Threshold Tuning (WITH PROGRESS)
# ============================================================
@torch.no_grad()
def evaluate_threshold(threshold, loader, model, verbose=False):
    model.eval()
    meter = SegmentationMeter()
    
    iterator = tqdm(loader, desc=f"Threshold {threshold:.2f}", leave=False) if verbose else loader
    
    for images, masks in iterator:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        
        with autocast_context():
            logits = model(images)
        
        probs = logits_to_probs(logits)
        preds = probs_to_preds(probs, threshold)
        
        meter.update(preds, masks)
        
        del images, masks, logits, probs, preds
    
    return meter.compute()

print("=" * 80)
print("Cell 10 - Threshold Tuning")
print("=" * 80)
print(f"Testing {len(THRESHOLD_GRID)} thresholds on {len(test_loader)} batches...")
print(f"Total test samples: {len(test_ds)}")
print()

threshold_results = []

for idx, threshold in enumerate(THRESHOLD_GRID):
    print(f"[{idx+1}/{len(THRESHOLD_GRID)}] Evaluating threshold {threshold:.2f}...", end=" ")
    
    val_metrics = evaluate_threshold(threshold, test_loader, model, verbose=False)
    
    threshold_results.append({
        "threshold": threshold,
        "iou": val_metrics["iou"],
        "dice": val_metrics["dice"],
        "accuracy": val_metrics["accuracy"],
        "precision": val_metrics["precision"],
        "recall": val_metrics["recall"],
        "f2": val_metrics["f2"]
    })
    
    print(f"IoU={val_metrics['iou']:.4f}, Dice={val_metrics['dice']:.4f}")

threshold_df = pd.DataFrame(threshold_results)
threshold_df.to_csv(os.path.join(OUTPUT_DIR, "threshold_tuning_results.csv"), index=False)

print()
print("=" * 80)
print("Threshold Tuning Results")
print("=" * 80)

best_threshold_iou = threshold_df.loc[threshold_df["iou"].idxmax(), "threshold"]
best_threshold_dice = threshold_df.loc[threshold_df["dice"].idxmax(), "threshold"]
best_threshold_f2 = threshold_df.loc[threshold_df["f2"].idxmax(), "threshold"]

print(f"Best threshold by IoU:  {best_threshold_iou:.2f} (IoU={threshold_df[threshold_df['threshold']==best_threshold_iou]['iou'].values[0]:.4f})")
print(f"Best threshold by Dice: {best_threshold_dice:.2f} (Dice={threshold_df[threshold_df['threshold']==best_threshold_dice]['dice'].values[0]:.4f})")
print(f"Best threshold by F2:   {best_threshold_f2:.2f} (F2={threshold_df[threshold_df['threshold']==best_threshold_f2]['f2'].values[0]:.4f})")

selected_threshold = best_threshold_iou
print(f"\nSelected threshold for testing: {selected_threshold:.2f}")

print("\nGenerating threshold tuning plots...")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Threshold Tuning Analysis", fontsize=16, fontweight="bold")

metrics_to_plot = ["iou", "dice", "accuracy", "precision", "recall", "f2"]
for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 3, idx % 3]
    ax.plot(threshold_df["threshold"], threshold_df[metric], marker="o", linewidth=2, color="blue")
    best_idx = threshold_df[metric].idxmax()
    ax.scatter(threshold_df.loc[best_idx, "threshold"], threshold_df.loc[best_idx, metric],
               color="red", s=200, zorder=5, label=f"Best: {threshold_df.loc[best_idx, 'threshold']:.2f}")
    ax.set_xlabel("Threshold", fontsize=11)
    ax.set_ylabel(metric.upper(), fontsize=11)
    ax.set_title(f"{metric.upper()} vs Threshold", fontsize=12, fontweight="bold")
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
threshold_plot_path = os.path.join(OUTPUT_DIR, "threshold_tuning_analysis.png")
plt.savefig(threshold_plot_path, dpi=120)
plt.close()
print(f"Threshold plot saved: {threshold_plot_path}")

print("\nThreshold tuning complete!")
print()

gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


Cell 10 - Threshold Tuning
Testing 17 thresholds on 287 batches...
Total test samples: 2295

[1/17] Evaluating threshold 0.10... IoU=0.6085, Dice=0.7566
[2/17] Evaluating threshold 0.15... IoU=0.6284, Dice=0.7718
[3/17] Evaluating threshold 0.20... IoU=0.6404, Dice=0.7808
[4/17] Evaluating threshold 0.25... IoU=0.6492, Dice=0.7873
[5/17] Evaluating threshold 0.30... IoU=0.6564, Dice=0.7925
[6/17] Evaluating threshold 0.35... IoU=0.6629, Dice=0.7973
[7/17] Evaluating threshold 0.40... IoU=0.6682, Dice=0.8011
[8/17] Evaluating threshold 0.45... IoU=0.6726, Dice=0.8043
[9/17] Evaluating threshold 0.50... IoU=0.6768, Dice=0.8073
[10/17] Evaluating threshold 0.55... IoU=0.6798, Dice=0.8094
[11/17] Evaluating threshold 0.60... IoU=0.6819, Dice=0.8109
[12/17] Evaluating threshold 0.65... IoU=0.6828, Dice=0.8115
[13/17] Evaluating threshold 0.70... IoU=0.6826, Dice=0.8114
[14/17] Evaluating threshold 0.75... IoU=0.6803, Dice=0.8097
[15/17] Evaluating threshold 0.80... IoU=0.6748, Dice=0.8058
[

In [17]:
##%% ============================================================
# Cell 11 - Post Processing Helpers
# ============================================================
def post_process_mask(mask, min_area=25, kernel_size=3):
    mask = mask.astype(np.uint8)
    
    if kernel_size > 1 and CV2_AVAILABLE:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    
    if CV2_AVAILABLE and min_area > 0:
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask)
        
        for label_id in range(1, num_labels):
            if stats[label_id, cv2.CC_STAT_AREA] < min_area:
                mask[labels == label_id] = 0
    
    return mask

def post_process_prediction(pred_mask, apply_post=True, min_area=25, kernel_size=3):
    if apply_post:
        return post_process_mask(pred_mask, min_area=min_area, kernel_size=kernel_size)
    return pred_mask

print("=" * 80)
print("Cell 11 - Post Processing")
print("=" * 80)
print(f"Post-processing enabled: {cfg.POST_PROCESS}")


Cell 11 - Post Processing
Post-processing enabled: True


In [21]:
##%% ============================================================
# Cell 12 - Test Time Augmentation (FIXED)
# ============================================================

tta_transforms_list = [
    A.Compose([
        A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ]),
    A.Compose([
        A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
        A.HorizontalFlip(p=1.0),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ]),
    A.Compose([
        A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
        A.VerticalFlip(p=1.0),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ]),
    A.Compose([
        A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
        A.RandomRotate90(p=1.0),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ]),
]

@torch.no_grad()
def predict_with_tta(image_path, model, use_tta=True, threshold=0.5):
    image = np.array(Image.open(image_path).convert("RGB"))
    h_orig, w_orig = image.shape[:2]
    
    num_transforms = len(tta_transforms_list) if use_tta else 1
    
    prob_map = np.zeros((cfg.IMG_SIZE, cfg.IMG_SIZE), dtype=np.float32)
    
    for i in range(num_transforms):
        transform = tta_transforms_list[i]
        aug = transform(image=image)
        img_aug = aug["image"].unsqueeze(0).to(DEVICE)
        
        with autocast_context():
            logits = model(img_aug)
        
        probs = logits_to_probs(logits)[0].cpu().numpy()
        
        if i == 1:
            probs = np.fliplr(probs)
        elif i == 2:
            probs = np.flipud(probs)
        elif i == 3:
            probs = np.rot90(probs, k=-1)
        
        prob_map += probs
        del img_aug, logits, probs
    
    prob_map /= num_transforms
    
    prob_map_orig = np.array(
        Image.fromarray((prob_map * 255).astype(np.uint8)).resize(
            (w_orig, h_orig),
            Image.BILINEAR
        )
    ).astype(np.float32) / 255.0
    
    pred_mask = (prob_map_orig >= threshold).astype(np.uint8)
    
    if cfg.POST_PROCESS:
        pred_mask = post_process_prediction(
            pred_mask,
            apply_post=True,
            min_area=cfg.POST_MIN_AREA,
            kernel_size=cfg.POST_KERNEL_SIZE
        )
    
    return prob_map_orig, pred_mask

print("=" * 80)
print("Cell 12 - Test Time Augmentation (FIXED)")
print("=" * 80)
print("TTA enabled with 4 transformations (original, hflip, vflip, rot90)")


Cell 12 - Test Time Augmentation (FIXED)
TTA enabled with 4 transformations (original, hflip, vflip, rot90)


In [23]:
##%% ============================================================
# Cell 13 - Test Evaluation Function
# ============================================================
@torch.no_grad()
def evaluate_on_test_set(use_tta=True, threshold=None, save_predictions=True):
    if threshold is None:
        threshold = selected_threshold
    
    model.eval()
    
    test_results = []
    predictions_dir = os.path.join(OUTPUT_DIR, "test_predictions")
    
    if save_predictions:
        Path(predictions_dir).mkdir(parents=True, exist_ok=True)
    
    pbar = tqdm(enumerate(test_ds.samples), total=len(test_ds.samples), desc="Testing")
    
    for idx, sample in pbar:
        img_path = sample["image_path"]
        mask_path = sample["mask_path"]
        image_name = sample["image_name"]
        disease_name = sample.get("disease_name", "Unknown")
        
        if not os.path.exists(img_path):
            continue
        
        try:
            prob_map, pred_mask = predict_with_tta(
                img_path,
                model,
                use_tta=use_tta,
                threshold=threshold
            )
            
            if mask_path and isinstance(mask_path, str) and os.path.exists(mask_path):
                true_mask = np.array(Image.open(mask_path).convert("L"))
                true_mask = (true_mask > 0).astype(np.uint8)
                
                if true_mask.shape != pred_mask.shape:
                    true_mask = np.array(
                        Image.fromarray(true_mask).resize(
                            pred_mask.shape[::-1],
                            Image.NEAREST
                        )
                    )
                
                metrics = compute_per_sample_metrics(pred_mask, true_mask)
            else:
                metrics = {
                    "iou": 0.0,
                    "dice": 0.0,
                    "accuracy": 0.0,
                    "precision": 0.0,
                    "recall": 0.0,
                    "tp": 0,
                    "fp": 0,
                    "fn": 0,
                    "tn": 0,
                }
            
            test_results.append({
                "image_name": image_name,
                "disease_name": disease_name,
                "iou": metrics["iou"],
                "dice": metrics["dice"],
                "accuracy": metrics["accuracy"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "tp": metrics["tp"],
                "fp": metrics["fp"],
                "fn": metrics["fn"],
                "tn": metrics["tn"],
            })
            
            if save_predictions:
                pred_path = os.path.join(predictions_dir, image_name.replace(".", "_pred."))
                Image.fromarray((pred_mask * 255).astype(np.uint8)).save(pred_path)
        
        except Exception as e:
            print(f"Error processing {image_name}: {str(e)}")
            continue
    
    test_results_df = pd.DataFrame(test_results)
    test_results_df.to_csv(
        os.path.join(OUTPUT_DIR, "test_results_detailed.csv"),
        index=False
    )
    
    return test_results_df

print("=" * 80)
print("Cell 13 - Test Evaluation Function Ready")
print("=" * 80)


Cell 13 - Test Evaluation Function Ready


In [24]:
##%% ============================================================
# Cell 14 - Final Test Evaluation
# ============================================================
print("=" * 80)
print("Cell 14 - Final Test Evaluation")
print("=" * 80)

test_results_df = evaluate_on_test_set(
    use_tta=True,
    threshold=selected_threshold,
    save_predictions=True
)

print(f"Evaluated {len(test_results_df)} test samples")

overall_metrics = {
    "iou_mean": float(test_results_df["iou"].mean()),
    "iou_std": float(test_results_df["iou"].std()),
    "iou_min": float(test_results_df["iou"].min()),
    "iou_max": float(test_results_df["iou"].max()),
    "dice_mean": float(test_results_df["dice"].mean()),
    "dice_std": float(test_results_df["dice"].std()),
    "dice_min": float(test_results_df["dice"].min()),
    "dice_max": float(test_results_df["dice"].max()),
    "accuracy_mean": float(test_results_df["accuracy"].mean()),
    "accuracy_std": float(test_results_df["accuracy"].std()),
    "precision_mean": float(test_results_df["precision"].mean()),
    "precision_std": float(test_results_df["precision"].std()),
    "recall_mean": float(test_results_df["recall"].mean()),
    "recall_std": float(test_results_df["recall"].std()),
}

print()
print("Overall Test Metrics:")
print(f"IoU:       {overall_metrics['iou_mean']:.4f} +/- {overall_metrics['iou_std']:.4f} "
      f"(min: {overall_metrics['iou_min']:.4f}, max: {overall_metrics['iou_max']:.4f})")
print(f"Dice:      {overall_metrics['dice_mean']:.4f} +/- {overall_metrics['dice_std']:.4f} "
      f"(min: {overall_metrics['dice_min']:.4f}, max: {overall_metrics['dice_max']:.4f})")
print(f"Accuracy:  {overall_metrics['accuracy_mean']:.4f} +/- {overall_metrics['accuracy_std']:.4f}")
print(f"Precision: {overall_metrics['precision_mean']:.4f} +/- {overall_metrics['precision_std']:.4f}")
print(f"Recall:    {overall_metrics['recall_mean']:.4f} +/- {overall_metrics['recall_std']:.4f}")

with open(os.path.join(OUTPUT_DIR, "test_overall_metrics.json"), "w") as f:
    json.dump(overall_metrics, f, indent=4)


Cell 14 - Final Test Evaluation


Testing: 100%|██████████| 2295/2295 [06:03<00:00,  6.31it/s]

Evaluated 2295 test samples

Overall Test Metrics:
IoU:       0.6303 +/- 0.2493 (min: 0.0000, max: 0.9949)
Dice:      0.7391 +/- 0.2270 (min: 0.0000, max: 0.9974)
Accuracy:  0.9216 +/- 0.0998
Precision: 0.7891 +/- 0.2360
Recall:    0.7647 +/- 0.2421


In [25]:
##%% ============================================================
# Cell 15 - Single Image Prediction
# ============================================================
def predict_single_image(image_path, use_tta=True, threshold=None, display=True):
    if threshold is None:
        threshold = selected_threshold
    
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")
    
    print(f"Processing: {image_path}")
    
    prob_map, pred_mask = predict_with_tta(
        image_path,
        model,
        use_tta=use_tta,
        threshold=threshold
    )
    
    image = np.array(Image.open(image_path).convert("RGB"))
    overlay = make_overlay(image, pred_mask, color=(0, 255, 0), alpha=0.5)
    
    result = {
        "image_path": image_path,
        "image": image,
        "prob_map": prob_map,
        "pred_mask": pred_mask,
        "overlay": overlay,
        "threshold": threshold,
        "lesion_pixels": int(pred_mask.sum()),
        "lesion_coverage": float(pred_mask.sum() / pred_mask.size * 100),
    }
    
    if display:
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        
        axes[0].imshow(image)
        axes[0].set_title("Original Image")
        axes[0].axis("off")
        
        axes[1].imshow(prob_map, cmap="jet")
        axes[1].set_title(f"Probability Map (Threshold: {threshold:.2f})")
        axes[1].axis("off")
        
        axes[2].imshow(pred_mask, cmap="gray")
        axes[2].set_title(f"Predicted Mask\nLesion Coverage: {result['lesion_coverage']:.2f}%")
        axes[2].axis("off")
        
        axes[3].imshow(overlay)
        axes[3].set_title("Prediction Overlay")
        axes[3].axis("off")
        
        plt.tight_layout()
        
        output_path = os.path.join(
            OUTPUT_DIR,
            f"single_prediction_{Path(image_path).stem}.png"
        )
        plt.savefig(output_path, dpi=120)
        plt.close()
        
        print(f"Visualization saved: {output_path}")
    
    return result

print("=" * 80)
print("Cell 15 - Single Image Prediction Function Ready")
print("=" * 80)


Cell 15 - Single Image Prediction Function Ready


In [26]:
##%% ============================================================
# Cell 16 - Results Summary and Visualization
# ============================================================
print("=" * 80)
print("Cell 16 - Final Summary")
print("=" * 80)

summary_report = {
    "project_name": cfg.PROJECT_NAME,
    "dataset_name": cfg.DATASET_NAME,
    "model_backbone": "EfficientNet-B4",
    "model_architecture": "Attention U-Net",
    "image_size": cfg.IMG_SIZE,
    "num_test_samples": len(test_results_df),
    "threshold_selected": float(selected_threshold),
    "tta_enabled": True,
    "post_processing_enabled": cfg.POST_PROCESS,
    "test_metrics": overall_metrics,
    "trained_epochs": epoch_loaded,
}

with open(os.path.join(OUTPUT_DIR, "final_summary_report.json"), "w") as f:
    json.dump(summary_report, f, indent=4)

print("Final Summary Report:")
print(json.dumps(summary_report, indent=2))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Final Test Results Summary", fontsize=16, fontweight="bold")

axes[0, 0].hist(test_results_df["iou"], bins=30, color="blue", alpha=0.7, edgecolor="black")
axes[0, 0].axvline(test_results_df["iou"].mean(), color="red", linestyle="--", linewidth=2, label="Mean")
axes[0, 0].set_xlabel("IoU", fontsize=11)
axes[0, 0].set_ylabel("Frequency", fontsize=11)
axes[0, 0].set_title(f"IoU Distribution (Mean: {test_results_df['iou'].mean():.4f})", fontsize=12, fontweight="bold")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].hist(test_results_df["dice"], bins=30, color="green", alpha=0.7, edgecolor="black")
axes[0, 1].axvline(test_results_df["dice"].mean(), color="red", linestyle="--", linewidth=2, label="Mean")
axes[0, 1].set_xlabel("Dice", fontsize=11)
axes[0, 1].set_ylabel("Frequency", fontsize=11)
axes[0, 1].set_title(f"Dice Distribution (Mean: {test_results_df['dice'].mean():.4f})", fontsize=12, fontweight="bold")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

disease_iou = test_results_df.groupby("disease_name")["iou"].mean().sort_values(ascending=False).head(15)
axes[1, 0].barh(disease_iou.index, disease_iou.values, color="orange", edgecolor="black")
axes[1, 0].set_xlabel("Mean IoU", fontsize=11)
axes[1, 0].set_title("Top 15 Disease Categories by Mean IoU", fontsize=12, fontweight="bold")
axes[1, 0].grid(axis="x", alpha=0.3)

metrics_names = ["iou", "dice", "accuracy", "precision", "recall"]
metrics_means = [test_results_df[m].mean() for m in metrics_names]
metrics_stds = [test_results_df[m].std() for m in metrics_names]
colors_bars = ["blue", "green", "red", "orange", "purple"]

x_pos = np.arange(len(metrics_names))
axes[1, 1].bar(x_pos, metrics_means, yerr=metrics_stds, capsize=5, color=colors_bars, alpha=0.7, edgecolor="black")
axes[1, 1].set_ylabel("Score", fontsize=11)
axes[1, 1].set_title("Overall Metrics Performance", fontsize=12, fontweight="bold")
axes[1, 1].set_xticks(x_pos)
axes[1, 1].set_xticklabels([m.upper() for m in metrics_names])
axes[1, 1].set_ylim([0, 1.0])
axes[1, 1].grid(axis="y", alpha=0.3)

for i, (m, s) in enumerate(zip(metrics_means, metrics_stds)):
    axes[1, 1].text(i, m + s + 0.02, f"{m:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
summary_plot_path = os.path.join(OUTPUT_DIR, "final_test_summary.png")
plt.savefig(summary_plot_path, dpi=120)
plt.close()
print(f"Summary plot saved: {summary_plot_path}")


Cell 16 - Final Summary
Final Summary Report:
{
  "project_name": "Plant Diagnosis with Lesion Segmentation",
  "dataset_name": "PlantSeg",
  "model_backbone": "EfficientNet-B4",
  "model_architecture": "Attention U-Net",
  "image_size": 320,
  "num_test_samples": 2295,
  "threshold_selected": 0.65,
  "tta_enabled": true,
  "post_processing_enabled": true,
  "test_metrics": {
    "iou_mean": 0.6302514099406823,
    "iou_std": 0.24931355683104192,
    "iou_min": 0.0,
    "iou_max": 0.9948940237357059,
    "dice_mean": 0.7390971219876892,
    "dice_std": 0.2270182576435023,
    "dice_min": 0.0,
    "dice_max": 0.9974404774371259,
    "accuracy_mean": 0.9215885654919024,
    "accuracy_std": 0.09975167375913267,
    "precision_mean": 0.7891342464128466,
    "precision_std": 0.2359686099275082,
    "recall_mean": 0.7647460946664422,
    "recall_std": 0.24211465263607856
  },
  "trained_epochs": 18
}
Summary plot saved: /kaggle/working/final_test_summary.png


In [28]:
##%% ============================================================
# Cell 17 - Error Analysis
# ============================================================
def analyze_failures(results_df, failure_threshold_iou=0.50, num_to_show=8):
    failed_samples = results_df[results_df["iou"] < failure_threshold_iou].copy()
    failed_samples = failed_samples.sort_values("iou").head(num_to_show)
    
    if len(failed_samples) == 0:
        print(f"No failures found below IoU threshold {failure_threshold_iou}")
        return None
    
    print(f"Found {len(failed_samples)} samples with IoU < {failure_threshold_iou}")
    print()
    print(failed_samples[["image_name", "disease_name", "iou", "dice", "precision", "recall"]])
    
    ds_dict = {sample["image_name"]: sample for sample in test_ds.samples}
    
    fig, axes = plt.subplots(min(num_to_show, len(failed_samples)), 4, figsize=(20, 5 * min(num_to_show, len(failed_samples))))
    
    if len(failed_samples) == 1:
        axes = axes.reshape(1, -1)
    
    fig.suptitle(f"Error Analysis - Samples with IoU < {failure_threshold_iou}", 
                 fontsize=16, fontweight="bold")
    
    for row_idx, (_, result_row) in enumerate(failed_samples.iterrows()):
        if row_idx >= len(axes):
            break
        
        image_name = result_row["image_name"]
        
        if image_name not in ds_dict:
            continue
        
        sample = ds_dict[image_name]
        img_path = sample["image_path"]
        mask_path = sample["mask_path"]
        
        if not os.path.exists(img_path) or not mask_path or not os.path.exists(mask_path):
            continue
        
        try:
            image = np.array(Image.open(img_path).convert("RGB"))
            true_mask = np.array(Image.open(mask_path).convert("L"))
            true_mask = (true_mask > 0).astype(np.uint8)
            
            prob_map, pred_mask = predict_with_tta(
                img_path,
                model,
                use_tta=True,
                threshold=selected_threshold
            )
            
            if true_mask.shape != pred_mask.shape:
                true_mask = np.array(
                    Image.fromarray(true_mask).resize(
                        pred_mask.shape[::-1],
                        Image.NEAREST
                    )
                )
            
            error_map = np.zeros((*pred_mask.shape, 3), dtype=np.uint8)
            tp = (pred_mask == 1) & (true_mask == 1)
            fp = (pred_mask == 1) & (true_mask == 0)
            fn = (pred_mask == 0) & (true_mask == 1)
            
            error_map[tp] = [0, 255, 0]
            error_map[fp] = [255, 0, 0]
            error_map[fn] = [0, 0, 255]
            
            axes[row_idx, 0].imshow(image)
            axes[row_idx, 0].set_title(f"Image: {image_name[:20]}")
            axes[row_idx, 0].axis("off")
            
            axes[row_idx, 1].imshow(true_mask, cmap="gray")
            axes[row_idx, 1].set_title("Ground Truth")
            axes[row_idx, 1].axis("off")
            
            axes[row_idx, 2].imshow(pred_mask, cmap="gray")
            axes[row_idx, 2].set_title(f"Prediction\nIoU: {result_row['iou']:.3f}")
            axes[row_idx, 2].axis("off")
            
            axes[row_idx, 3].imshow(error_map)
            axes[row_idx, 3].set_title("Error Map\nGreen:TP Red:FP Blue:FN")
            axes[row_idx, 3].axis("off")
        
        except Exception as e:
            print(f"Error analyzing {image_name}: {str(e)}")
            continue
    
    plt.tight_layout()
    error_path = os.path.join(OUTPUT_DIR, "error_analysis.png")
    plt.savefig(error_path, dpi=120)
    plt.close()
    print(f"Error analysis plot saved: {error_path}")
    
    return failed_samples

print("=" * 80)
print("Cell 17 - Error Analysis")
print("=" * 80)

failure_analysis = analyze_failures(test_results_df, failure_threshold_iou=0.50, num_to_show=8)


Cell 17 - Error Analysis
Found 8 samples with IoU < 0.5

                                  image_name                   disease_name  \
1583            squash_powdery_mildew_37.jpg          squash powdery mildew   
1409               rice_sheath_blight_57.jpg             rice sheath blight   
2282      zucchini_yellow_mosaic_virus_9.jpg   zucchini yellow mosaic virus   
1614  strawberry_anthracnose_google_0007.jpg  strawberry anthracnose google   
1290     plum_pocket_disease_google_0028.jpg     plum pocket disease google   
1291     plum_pocket_disease_google_0029.jpg     plum pocket disease google   
1252                peach_scab_Bing_0007.jpg                peach scab Bing   
1886          wheat_head_scab_Baidu_0162.jpg          wheat head scab Baidu   

      iou  dice  precision  recall  
1583  0.0   0.0        0.0     0.0  
1409  0.0   0.0        0.0     0.0  
2282  0.0   0.0        0.0     0.0  
1614  0.0   0.0        0.0     0.0  
1290  0.0   0.0        0.0     0.0  
1291  0.0

In [29]:
##%% ============================================================
# Cell 18 - Results Visualization
# ============================================================
def visualize_test_results(num_samples=12):
    sample_indices = np.random.choice(len(test_ds.samples), size=min(num_samples, len(test_ds.samples)), replace=False)
    
    n_cols = 4
    n_rows = len(sample_indices)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
    
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    fig.suptitle("Test Set Predictions", fontsize=16, fontweight="bold")
    
    for row_idx, sample_idx in enumerate(sample_indices):
        sample = test_ds.samples[sample_idx]
        img_path = sample["image_path"]
        mask_path = sample["mask_path"]
        
        if not os.path.exists(img_path) or not mask_path or not os.path.exists(mask_path):
            continue
        
        try:
            image = np.array(Image.open(img_path).convert("RGB"))
            true_mask = np.array(Image.open(mask_path).convert("L"))
            true_mask = (true_mask > 0).astype(np.uint8)
            
            prob_map, pred_mask = predict_with_tta(
                img_path,
                model,
                use_tta=True,
                threshold=selected_threshold
            )
            
            if true_mask.shape != pred_mask.shape:
                true_mask = np.array(
                    Image.fromarray(true_mask).resize(
                        pred_mask.shape[::-1],
                        Image.NEAREST
                    )
                )
            
            metrics = compute_per_sample_metrics(pred_mask, true_mask)
            
            overlay_pred = make_overlay(image, pred_mask, color=(0, 255, 0), alpha=0.5)
            overlay_true = make_overlay(image, true_mask, color=(255, 0, 0), alpha=0.5)
            
            axes[row_idx, 0].imshow(image)
            axes[row_idx, 0].set_title("Image")
            axes[row_idx, 0].axis("off")
            
            axes[row_idx, 1].imshow(true_mask, cmap="gray")
            axes[row_idx, 1].set_title("Ground Truth")
            axes[row_idx, 1].axis("off")
            
            axes[row_idx, 2].imshow(pred_mask, cmap="gray")
            axes[row_idx, 2].set_title(f"Prediction\nIoU: {metrics['iou']:.3f}")
            axes[row_idx, 2].axis("off")
            
            axes[row_idx, 3].imshow(overlay_pred)
            axes[row_idx, 3].set_title(f"Dice: {metrics['dice']:.3f}")
            axes[row_idx, 3].axis("off")
        
        except Exception as e:
            print(f"Error visualizing sample {sample_idx}: {str(e)}")
            continue
    
    plt.tight_layout()
    viz_path = os.path.join(OUTPUT_DIR, "test_results_visualization.png")
    plt.savefig(viz_path, dpi=120)
    plt.close()
    print(f"Visualization saved: {viz_path}")

visualize_test_results(num_samples=12)

print("=" * 80)
print("Cell 18 - Results Visualization")
print("=" * 80)


Visualization saved: /kaggle/working/test_results_visualization.png
Cell 18 - Results Visualization


In [30]:
##%% ============================================================
# Cell 19 - Save Final Results
# ============================================================
print("=" * 80)
print("Cell 19 - Save Results")
print("=" * 80)

results_summary = {
    "test_set_size": len(test_results_df),
    "test_metrics": overall_metrics,
    "threshold_selected": float(selected_threshold),
    "tta_enabled": True,
    "post_processing_enabled": cfg.POST_PROCESS,
    "model_checkpoint": cfg.BEST_MODEL_PATH,
    "trained_epochs": int(epoch_loaded),
    "files_saved": {
        "test_results_detailed": os.path.join(OUTPUT_DIR, "test_results_detailed.csv"),
        "test_overall_metrics": os.path.join(OUTPUT_DIR, "test_overall_metrics.json"),
        "threshold_tuning_results": os.path.join(OUTPUT_DIR, "threshold_tuning_results.csv"),
        "threshold_tuning_analysis": os.path.join(OUTPUT_DIR, "threshold_tuning_analysis.png"),
        "test_predictions": os.path.join(OUTPUT_DIR, "test_predictions"),
        "final_test_summary": os.path.join(OUTPUT_DIR, "final_test_summary.png"),
        "test_results_visualization": os.path.join(OUTPUT_DIR, "test_results_visualization.png"),
        "error_analysis": os.path.join(OUTPUT_DIR, "error_analysis.png"),
        "final_summary_report": os.path.join(OUTPUT_DIR, "final_summary_report.json"),
    }
}

with open(os.path.join(OUTPUT_DIR, "testing_phase_summary.json"), "w") as f:
    json.dump(results_summary, f, indent=4)

print("All results saved successfully")
print()
print("Files saved:")
for key, value in results_summary["files_saved"].items():
    print(f"  {key}: {value}")

print()
print("Final Test Metrics:")
for key, value in overall_metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

print()
print("Testing phase complete!")


Cell 19 - Save Results
All results saved successfully

Files saved:
  test_results_detailed: /kaggle/working/test_results_detailed.csv
  test_overall_metrics: /kaggle/working/test_overall_metrics.json
  threshold_tuning_results: /kaggle/working/threshold_tuning_results.csv
  threshold_tuning_analysis: /kaggle/working/threshold_tuning_analysis.png
  test_predictions: /kaggle/working/test_predictions
  final_test_summary: /kaggle/working/final_test_summary.png
  test_results_visualization: /kaggle/working/test_results_visualization.png
  error_analysis: /kaggle/working/error_analysis.png
  final_summary_report: /kaggle/working/final_summary_report.json

Final Test Metrics:
  iou_mean: 0.6303
  iou_std: 0.2493
  iou_min: 0.0000
  iou_max: 0.9949
  dice_mean: 0.7391
  dice_std: 0.2270
  dice_min: 0.0000
  dice_max: 0.9974
  accuracy_mean: 0.9216
  accuracy_std: 0.0998
  precision_mean: 0.7891
  precision_std: 0.2360
  recall_mean: 0.7647
  recall_std: 0.2421

Testing phase complete!


In [31]:
##%% ============================================================
# Cell 20 - Display Sample Predictions
# ============================================================
print("=" * 80)
print("Cell 20 - Display Sample Predictions")
print("=" * 80)

num_samples_to_show = 12
sample_indices = np.random.choice(len(test_ds), size=min(num_samples_to_show, len(test_ds)), replace=False)

fig, axes = plt.subplots(num_samples_to_show, 4, figsize=(20, 5 * num_samples_to_show))

if num_samples_to_show == 1:
    axes = axes.reshape(1, -1)

fig.suptitle("Sample Test Set Predictions", fontsize=18, fontweight="bold")

for row_idx, sample_idx in enumerate(sample_indices):
    sample = test_ds.samples[sample_idx]
    img_path = sample["image_path"]
    mask_path = sample["mask_path"]
    disease_name = sample["disease_name"]
    image_name = sample["image_name"]
    
    if not os.path.exists(img_path):
        continue
    
    try:
        image = np.array(Image.open(img_path).convert("RGB"))
        
        prob_map, pred_mask = predict_with_tta(
            img_path,
            model,
            use_tta=True,
            threshold=selected_threshold
        )
        
        overlay_pred = make_overlay(image, pred_mask, color=(0, 255, 0), alpha=0.5)
        
        axes[row_idx, 0].imshow(image)
        axes[row_idx, 0].set_title(f"Original Image\n{image_name[:30]}", fontsize=10, fontweight="bold")
        axes[row_idx, 0].axis("off")
        
        axes[row_idx, 1].imshow(prob_map, cmap="jet")
        axes[row_idx, 1].set_title(f"Probability Map\nThreshold: {selected_threshold:.2f}", fontsize=10, fontweight="bold")
        axes[row_idx, 1].axis("off")
        
        axes[row_idx, 2].imshow(pred_mask, cmap="gray")
        lesion_coverage = float(pred_mask.sum() / pred_mask.size * 100)
        axes[row_idx, 2].set_title(f"Predicted Mask\nLesion: {lesion_coverage:.2f}%", fontsize=10, fontweight="bold")
        axes[row_idx, 2].axis("off")
        
        axes[row_idx, 3].imshow(overlay_pred)
        axes[row_idx, 3].set_title(f"Prediction Overlay\n{disease_name[:25]}", fontsize=10, fontweight="bold")
        axes[row_idx, 3].axis("off")
        
    except Exception as e:
        print(f"Error visualizing sample {sample_idx}: {str(e)}")
        continue

plt.tight_layout()
predictions_plot_path = os.path.join(OUTPUT_DIR, "sample_predictions.png")
plt.savefig(predictions_plot_path, dpi=120, bbox_inches='tight')
plt.close()

print(f"\nSample predictions visualization saved: {predictions_plot_path}")
print(f"Total samples shown: {num_samples_to_show}")


Cell 20 - Display Sample Predictions

Sample predictions visualization saved: /kaggle/working/sample_predictions.png
Total samples shown: 12


In [32]:
##%% ============================================================
# Cell 20 - Display Sample Predictions with Ground Truth
# ============================================================
print("=" * 80)
print("Cell 20 - Display Sample Predictions with Ground Truth")
print("=" * 80)

num_samples_to_show = 8
sample_indices = np.random.choice(len(test_ds), size=min(num_samples_to_show, len(test_ds)), replace=False)

fig, axes = plt.subplots(num_samples_to_show, 5, figsize=(25, 5 * num_samples_to_show))

if num_samples_to_show == 1:
    axes = axes.reshape(1, -1)

fig.suptitle("Sample Test Predictions with Ground Truth Comparison", fontsize=18, fontweight="bold")

for row_idx, sample_idx in enumerate(sample_indices):
    sample = test_ds.samples[sample_idx]
    img_path = sample["image_path"]
    mask_path = sample["mask_path"]
    disease_name = sample["disease_name"]
    image_name = sample["image_name"]
    
    if not os.path.exists(img_path):
        continue
    
    try:
        image = np.array(Image.open(img_path).convert("RGB"))
        
        prob_map, pred_mask = predict_with_tta(
            img_path,
            model,
            use_tta=True,
            threshold=selected_threshold
        )
        
        overlay_pred = make_overlay(image, pred_mask, color=(0, 255, 0), alpha=0.5)
        
        axes[row_idx, 0].imshow(image)
        axes[row_idx, 0].set_title(f"Original Image\n{image_name[:25]}", fontsize=9, fontweight="bold")
        axes[row_idx, 0].axis("off")
        
        if mask_path and os.path.exists(mask_path):
            true_mask = np.array(Image.open(mask_path).convert("L"))
            true_mask = (true_mask > 0).astype(np.uint8)
            
            if true_mask.shape != pred_mask.shape:
                true_mask = np.array(Image.fromarray(true_mask).resize(pred_mask.shape[::-1], Image.NEAREST))
            
            metrics = compute_per_sample_metrics(pred_mask, true_mask)
            
            axes[row_idx, 1].imshow(true_mask, cmap="gray")
            axes[row_idx, 1].set_title(f"Ground Truth\nMask", fontsize=9, fontweight="bold")
            axes[row_idx, 1].axis("off")
        else:
            axes[row_idx, 1].text(0.5, 0.5, "No GT Mask", ha='center', va='center', fontsize=12)
            axes[row_idx, 1].set_title("Ground Truth", fontsize=9, fontweight="bold")
            axes[row_idx, 1].axis("off")
            metrics = None
        
        axes[row_idx, 2].imshow(prob_map, cmap="jet")
        axes[row_idx, 2].set_title(f"Probability Map\nThreshold: {selected_threshold:.2f}", fontsize=9, fontweight="bold")
        axes[row_idx, 2].axis("off")
        
        axes[row_idx, 3].imshow(pred_mask, cmap="gray")
        lesion_coverage = float(pred_mask.sum() / pred_mask.size * 100)
        if metrics:
            title_text = f"Prediction\nIoU: {metrics['iou']:.3f}"
        else:
            title_text = f"Prediction\nLesion: {lesion_coverage:.2f}%"
        axes[row_idx, 3].set_title(title_text, fontsize=9, fontweight="bold")
        axes[row_idx, 3].axis("off")
        
        axes[row_idx, 4].imshow(overlay_pred)
        if metrics:
            title_text = f"Overlay\nDice: {metrics['dice']:.3f}"
        else:
            title_text = f"Overlay\n{disease_name[:20]}"
        axes[row_idx, 4].set_title(title_text, fontsize=9, fontweight="bold")
        axes[row_idx, 4].axis("off")
        
    except Exception as e:
        print(f"Error visualizing sample {sample_idx}: {str(e)}")
        continue

plt.tight_layout()
predictions_plot_path = os.path.join(OUTPUT_DIR, "sample_predictions_with_gt.png")
plt.savefig(predictions_plot_path, dpi=120, bbox_inches='tight')
plt.close()

print(f"\nSample predictions visualization saved: {predictions_plot_path}")
print(f"Total samples shown: {num_samples_to_show}")
print("\nColumns:")
print("  1. Original Image")
print("  2. Ground Truth Mask (if available)")
print("  3. Probability Map")
print("  4. Predicted Mask")
print("  5. Prediction Overlay")


Cell 20 - Display Sample Predictions with Ground Truth

Sample predictions visualization saved: /kaggle/working/sample_predictions_with_gt.png
Total samples shown: 8

Columns:
  1. Original Image
  2. Ground Truth Mask (if available)
  3. Probability Map
  4. Predicted Mask
  5. Prediction Overlay


In [33]:
##%% ============================================================
# Cell 20 - Display Best and Worst Predictions
# ============================================================
print("=" * 80)
print("Cell 20 - Display Best and Worst Predictions")
print("=" * 80)

if len(test_results_df) == 0:
    print("No test results to display")
else:
    best_samples = test_results_df.nlargest(6, 'iou')
    worst_samples = test_results_df.nsmallest(6, 'iou')
    
    fig = plt.figure(figsize=(24, 14))
    gs = fig.add_gridspec(4, 6, hspace=0.4, wspace=0.3)
    
    fig.suptitle("Best and Worst Segmentation Predictions", fontsize=20, fontweight="bold")
    
    ds_dict = {sample["image_name"]: sample for sample in test_ds.samples}
    
    for idx, (_, result_row) in enumerate(best_samples.iterrows()):
        if idx >= 6:
            break
        
        image_name = result_row["image_name"]
        if image_name not in ds_dict:
            continue
        
        sample = ds_dict[image_name]
        img_path = sample["image_path"]
        mask_path = sample["mask_path"]
        
        if not os.path.exists(img_path) or not mask_path or not os.path.exists(mask_path):
            continue
        
        try:
            image = np.array(Image.open(img_path).convert("RGB"))
            true_mask = np.array(Image.open(mask_path).convert("L"))
            true_mask = (true_mask > 0).astype(np.uint8)
            
            prob_map, pred_mask = predict_with_tta(img_path, model, use_tta=True, threshold=selected_threshold)
            
            if true_mask.shape != pred_mask.shape:
                true_mask = np.array(Image.fromarray(true_mask).resize(pred_mask.shape[::-1], Image.NEAREST))
            
            ax = fig.add_subplot(gs[0, idx])
            ax.imshow(image)
            ax.set_title(f"Best #{idx+1}\nIoU: {result_row['iou']:.3f}", fontsize=11, fontweight="bold", color="green")
            ax.axis("off")
            
        except Exception as e:
            print(f"Error with best sample {idx}: {str(e)}")
    
    for idx, (_, result_row) in enumerate(worst_samples.iterrows()):
        if idx >= 6:
            break
        
        image_name = result_row["image_name"]
        if image_name not in ds_dict:
            continue
        
        sample = ds_dict[image_name]
        img_path = sample["image_path"]
        mask_path = sample["mask_path"]
        
        if not os.path.exists(img_path) or not mask_path or not os.path.exists(mask_path):
            continue
        
        try:
            image = np.array(Image.open(img_path).convert("RGB"))
            true_mask = np.array(Image.open(mask_path).convert("L"))
            true_mask = (true_mask > 0).astype(np.uint8)
            
            prob_map, pred_mask = predict_with_tta(img_path, model, use_tta=True, threshold=selected_threshold)
            
            if true_mask.shape != pred_mask.shape:
                true_mask = np.array(Image.fromarray(true_mask).resize(pred_mask.shape[::-1], Image.NEAREST))
            
            ax = fig.add_subplot(gs[1, idx])
            ax.imshow(image)
            ax.set_title(f"Worst #{idx+1}\nIoU: {result_row['iou']:.3f}", fontsize=11, fontweight="bold", color="red")
            ax.axis("off")
            
        except Exception as e:
            print(f"Error with worst sample {idx}: {str(e)}")
    
    plt.savefig(os.path.join(OUTPUT_DIR, "best_and_worst_predictions.png"), dpi=120, bbox_inches='tight')
    plt.close()
    
    print(f"Best and worst predictions saved: {os.path.join(OUTPUT_DIR, 'best_and_worst_predictions.png')}")
    print("\nBest 6 Predictions:")
    print(best_samples[["image_name", "disease_name", "iou", "dice"]].to_string())
    print("\nWorst 6 Predictions:")
    print(worst_samples[["image_name", "disease_name", "iou", "dice"]].to_string())


Cell 20 - Display Best and Worst Predictions
Best and worst predictions saved: /kaggle/working/best_and_worst_predictions.png

Best 6 Predictions:
                                                   image_name                disease_name       iou      dice
454                        cabbage_downy_mildew_Bing_0163.jpg   cabbage downy mildew Bing  0.994894  0.997440
2025                      wheat_powdery_mildew_Baidu_0286.jpg  wheat powdery mildew Baidu  0.992698  0.996335
158   banana_black_leaf_streak_banana black sigatoka (77).jpg    banana black leaf streak  0.990641  0.995299
266                                         bean_rust_178.jpg                   bean rust  0.989792  0.994870
1308                                  plum_rust_Bing_0024.jpg              plum rust Bing  0.989640  0.994793
730                       corn_gray_leaf_spot_google_0172.jpg  corn gray leaf spot google  0.988355  0.994143

Worst 6 Predictions:
                                   image_name                